# CLIQUE — Online Retail II

Notebook walkthrough for the pipeline in `pipelines/` and `clique/`.

- **Raw:** `data/raw/online_retail_ii.xlsx` only
- **Processed:** CSV caches in `data/processed/`
- **Metrics:** intrinsic only (silhouette, Davies–Bouldin, Calinski–Harabasz)

In [ ]:
# 1. Imports
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import config
from clique.algorithm import CLIQUE
from pipelines import benchmark, data, preprocess, train
from pipelines.benchmark import run_baseline_comparison
from pipelines.data import build_customer_profiles, clean_data, load_raw_data

sns.set_theme(style="whitegrid")
print("ROOT:", ROOT)

## 2. Load workbook and build customer profiles

In [ ]:
path = data.resolve_retail_xlsx()
profiles = data.build_profiles_from_xlsx(path, save_artifacts=True)
profiles.head()

## 3. EDA — histograms, boxplots, correlation

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(15, 7))
for ax, col in zip(axes.ravel(), config.FEATURE_NAMES):
    profiles[col].hist(ax=ax, bins=30)
    ax.set_title(col)
plt.tight_layout(); plt.show()

plt.figure(figsize=(12, 5))
sns.boxplot(data=profiles[config.FEATURE_NAMES], orient="h")
plt.title("Feature boxplots (raw scale)"); plt.tight_layout(); plt.show()

plt.figure(figsize=(8, 6))
sns.heatmap(profiles[config.FEATURE_NAMES].corr(), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Feature correlation"); plt.show()

print("Skewness (|skew|>1 => log-transform candidate):")
print(profiles[config.FEATURE_NAMES].skew().round(2).to_string())

## 4. Preprocess — log-transform, split (random_state=42), MinMax scale

In [ ]:
prep = preprocess.run_retail(profiles, winsorize=True)
X_train = prep["X_train"]
print("Scaled train range:", X_train.min(), X_train.max())

## 5. CLIQUE grid search over (xi, tau)
Selection: composite quality score (silhouette, coverage, Davies–Bouldin).

In [ ]:
grid = train.grid_search(X_train)
display(grid.sort_values("quality_score", ascending=False))

## 6. Train final model + save artifacts

In [ ]:
model, cluster_desc, _ = train.run_retail(X_train, do_grid_search=True)
print(f"{len(model.clusters_)} clusters across {len(model.subspace_coverage_)} subspaces")

## 7. Evaluate — comparison table + figures (CSV/PNG to results/)

In [ ]:
comparison = benchmark.run_retail(model, cluster_desc)
display(comparison)

## 8. One-shot pipeline (optional)
Equivalent to `python src/pipelines/run.py retail`.

In [ ]:
from pipelines import retail

# retail.run_from_xlsx(save_artifacts=True, do_grid_search=True)